# Stage 4b — Speaker attribution

**Attach:** `sarvam-diar-code`, `sarvam-diar-stage3`, `sarvam-diar-asr-indic`,
`sarvam-diar-asr-whisper`.
**Settings:** accelerator **None** — this stage is CPU-only and finishes in
seconds. Running it on a GPU session spends quota on nothing.

Stage 4a produced words with timestamps and no speaker. This stage crosses those
words with **one** diarization hypothesis at a time and writes one directory per
`(asr, diar)` condition. Because the words are identical across conditions, a
cpWER difference between two of them is attributable to the labelling — which is
the entire reason ASR and attribution are separate stages.

No audio is needed here, so the audio dataset stays detached.

### The four rules, and why each one is a choice

1. **Maximum overlap.** A word goes to the turn sharing the most time with it.
   Assigning by midpoint is cheaper and throws away exactly the information that
   matters on words straddling a boundary. Equal overlap breaks toward the
   earlier turn, so the output is deterministic.
2. **Orphans are kept, not dropped.** A word landing where the diarizer heard
   nothing is given the nearest turn and flagged. Dropping it would delete it
   from the hypothesis and register as a cpWER deletion — quietly *rewarding* a
   system for missing speech. The flag is what lets Stage 6 separate this rule's
   cost from real labelling errors.
3. **Contested words are counted in two buckets.** `overlap` means two speakers
   genuinely active at once; `boundary` means a word crossing between two
   disjoint turns. Merging them would bury the first: the corpus is 7.60%
   overlapped, while every turn change makes boundary words.
4. **`--diar ref` is an oracle.** It attributes with the reference RTTM, giving a
   cpWER floor where labelling is perfect by construction, so every other
   condition reads as "ASR error + what this diarizer cost". Diagnostic only —
   nothing from it is fed back to any model, and it is labelled `oracle` in
   every table.

In [1]:
import pathlib, shutil, json

ROOT = pathlib.Path("/kaggle/input")
CODE = next(p.parent for p in ROOT.rglob("stage4_attribute.py"))
WORK = pathlib.Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)

for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")

# Every attached dataset that carries a data/ tree is merged into one working
# copy: Stage 3's RTTMs, and one directory per ASR system from the two Stage 4a
# datasets. No audio is needed, so that dataset stays detached.
for src in sorted(ROOT.rglob("data")):
    if src.is_dir() and any((src / d).exists() for d in ("hyp", "ref", "asr")):
        shutil.copytree(src, WORK, dirs_exist_ok=True)
        print("restored", src)

print()
print("CODE:", CODE)
ASR = sorted(p.name for p in (WORK / "asr").glob("*") if p.is_dir())
DIAR = sorted(p.name for p in (WORK / "hyp").glob("*") if p.is_dir())
for a in ASR:
    n = len(list((WORK / "asr" / a / "words").glob("*.json")))
    print(f"  asr/{a:16} {n:3} clips")
for d in DIAR:
    print(f"  hyp/{d:16} {len(list((WORK / 'hyp' / d / 'rttm').glob('*.rttm'))):3} rttm")
print(f"  ref/rttm{'':12} {len(list((WORK / 'ref' / 'rttm').glob('*.rttm'))):3} rttm")
print()
print("ASR :", ASR)
print("DIAR:", DIAR, "+ ref (oracle)")
assert ASR, "no ASR words found -- attach the Stage 4a dataset(s)"

# Built here rather than interpolated into the shell line below: IPython's {}
# expansion chokes on quotes inside the braces.
ASR_ARG, DIAR_ARG = " ".join(ASR), " ".join(DIAR)

restored /kaggle/input/datasets/ritankarmondal/sarvam-diar-asr-indic/data
restored /kaggle/input/datasets/ritankarmondal/sarvam-diar-stage3/data

CODE: /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
  asr/indicconformer    99 clips
  hyp/pyannote31        99 rttm
  hyp/sortformer        74 rttm
  hyp/sortformer_stream  99 rttm
  ref/rttm             100 rttm

ASR : ['indicconformer']
DIAR: ['pyannote31', 'sortformer', 'sortformer_stream'] + ref (oracle)


### What must be true before running

Attribution is silent about inputs it never sees: an ASR system whose dataset is
not attached simply produces no conditions, and the run still exits 0. The cell
above prints the inventory so a missing attachment is caught here rather than
discovered as a hole in the results table.

`sortformer` has 74 of 99 RTTMs — the long clips OOM'd in Stage 3. Those 25 clips
are expected to **fail loudly** below rather than be written as empty, which is
why the script's exit code is non-zero on that condition. That is the correct
outcome, not a bug to work around: a clip with no hypothesis is not a clip where
nobody spoke.

In [2]:
!python stage4_attribute.py --asr {ASR_ARG} --diar {DIAR_ARG} ref --data data


[cond] indicconformer__pyannote31
[plan] 99 clips: 0 done, 99 pending, running 99 now
[done] ok=99 fail=0  64,064 words, 1,795 orphaned (2.80%), 2,659 in overlapped speech (4.15%), 837 on a turn boundary (1.31%)

[cond] indicconformer__sortformer
[plan] 99 clips: 0 done, 99 pending, running 99 now
  fail: 0esIFSOAcFs__000000000_000913000  ValueError: no turns in RTTM
  fail: 0p6cktLGIfY__000012000_000930000  ValueError: no turns in RTTM
  fail: 13VBh0Z6QmE__000000000_000904000  ValueError: no turns in RTTM
  fail: 7WhVNRHbjIY__000091000_001089000  ValueError: no turns in RTTM
  fail: 7k-xDqNjESc__000073000_000981000  ValueError: no turns in RTTM
  fail: 83gP2vLH7UY__000255000_002005000  ValueError: no turns in RTTM
  fail: 86mMTUeDiR8__000181000_001079000  ValueError: no turns in RTTM
  fail: 8mvORuRHw2U__000000000_001197000  ValueError: no turns in RTTM
  fail: 8o7jKmq6HsM__000030000_001770000  ValueError: no turns in RTTM
  fail: BGAAfht5dYw__000000000_000612000  ValueError: no turn

### Read the orphan rate before believing any cpWER

If a system orphans a few percent of words, rule 2 is a footnote. If it orphans
twenty, the rule is doing heavy lifting and the writeup has to say so before
quoting a single number.

In [3]:
import json, pathlib
import pandas as pd

rows = []
for cond in sorted((pathlib.Path("/kaggle/working/data/attrib")).glob("*")):
    mf = cond / "manifest.jsonl"
    if not mf.exists():
        continue
    recs = [json.loads(l) for l in mf.read_text().splitlines() if l.strip()]
    ok = [r for r in recs if r["status"] == "ok"]
    w = sum(r["n_words"] for r in ok) or 1
    asr, diar = cond.name.split("__")
    rows.append({
        "asr": asr,
        "diar": diar + (" (oracle)" if diar == "ref" else ""),
        "clips_ok": len(ok),
        "clips_fail": len(recs) - len(ok),
        "words": sum(r["n_words"] for r in ok),
        "orphan_%": round(100 * sum(r["n_orphan"] for r in ok) / w, 2),
        "overlap_%": round(100 * sum(r["n_overlap_words"] for r in ok) / w, 2),
        "boundary_%": round(100 * sum(r["n_boundary_words"] for r in ok) / w, 2),
        "mean_spk": round(sum(r["n_speakers"] for r in ok) / max(len(ok), 1), 2),
    })

df = pd.DataFrame(rows).sort_values(["asr", "diar"])
print(df.to_string(index=False))

# One clip, end to end, so the words are visibly attached to speakers.
cond = sorted(pathlib.Path("/kaggle/working/data/attrib").glob("*__ref"))[0]
clip = sorted(cond.glob("*.json"))[0]
d = json.load(open(clip, encoding="utf-8"))
print()
print(f"{d['clip_id'][:44]}  {d['lang']}  {d['n_words']} words, "
      f"{len(d['speakers'])} speakers, {d['n_orphan']} orphaned")
for spk, text in d["by_speaker"].items():
    print(f"  {spk:12} {text[:110]}")

           asr              diar  clips_ok  clips_fail  words  orphan_%  overlap_%  boundary_%  mean_spk
indicconformer        pyannote31        99           0  64064      2.80       4.15        1.31      3.38
indicconformer      ref (oracle)        99           0  64064      2.36       8.92        1.21      3.52
indicconformer        sortformer        74          25  28554      3.08       2.46        1.72      2.85
indicconformer sortformer_stream        99           0  64064      1.87       3.55        1.44      3.20

0AEEA8NyVwY__000011000_000609000  mr  961 words, 3 speakers, 24 orphaned
  Speaker_A    ನमस्कार मी ଗౌरਵ जोशीી आणिनी तुमच سगळ्यां ಕॉી క୍ରꯦणीꯦ స్ مो আता ಅᱢोલ ने سांगितलं कि आणिनी तू पण سांगितलं कि गे 
  Speaker_B    मी ಅमोोल कꯔꯍडकर आणि ꯃै্ৰିꯅോ ଆज आप हा अज एक سीसीबीਕੇꯦ ಸ್पेشલचा ഭાગ घेऊन तुुमच्या സमोर येत आहेत आणिनी सીसीबीਕੇे 
  Speaker_C    धधନ୍ୟ्यवा سगळ्या प्रেಕ್ಷकांना ही माझا धधନ୍ୟ्यवाद ಅಮोोल आणि ଗौरਵ तुुम्ही ꯆांगل काम کرतायख ಪ್ರಶिकଷक म्हण मी गेଲେ


## Save

Output tab → **New dataset**, `sarvam-diar-attrib`. Stage 4c (scoring) attaches
it and never re-runs this.